# Notebook 1 — Build the Network
**YTS+ DSEB 2026 · Project B · Indian Railways**

---

Indian Railways runs over 13,000 trains a day across more than 8,000 stations.
Every train has a public schedule — a list of stations it stops at, with times.

Someone collected every schedule and put it online. We downloaded it.
By the end of this session you will have built a map of every physical railway track in India — using nothing but train timetables.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs('images', exist_ok=True)
DATA = 'data/'
print('Ready.')

---
## Part 1 — What does the raw data look like?

The original file is `schedules.json` — **417,080 rows**. Each row is one station stop for one train.

Here are 6 rows from it. This is train 12301 — the Howrah Rajdhani Express, New Delhi → Kolkata:

| train_number | station_code | departure | day | sequence |
|---|---|---|---|---|
| 12301 | NDLS (New Delhi) | 16:55 | 1 | 1 |
| 12301 | CNB (Kanpur) | 22:40 | 1 | 2 |
| 12301 | ALD (Prayagraj) | 01:15 | 2 | 3 |
| 12301 | MGS (Mughal Sarai) | 03:10 | 2 | 4 |
| 12301 | PNBE (Patna) | 07:00 | 2 | 5 |
| 12301 | HWH (Howrah) | 10:30 | 2 | 6 |

**One row = one train halting at one station.** That is the entire raw material.

From this we can extract: for each train, which two stations appear as consecutive stops? That is the connection between them.

### The stations

In [ ]:
# Each row is one station. Columns: code, name, state, lat (latitude), lon (longitude)
stations = pd.read_csv(DATA + 'stations_metrics.csv')

print(f'Total stations: {len(stations):,}')
print()
stations[['code','name','state','lat','lon']].head(10)

In [ ]:
# Which states have the most stations?
by_state = stations[stations['state'] != ''].groupby('state').size()
by_state = by_state.sort_values(ascending=False)

print('Stations per state (top 15):')
print(by_state.head(15).to_string())

In [ ]:
# Plot every station as a dot at its GPS location
fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter(stations['lon'], stations['lat'], s=0.4, color='steelblue', alpha=0.4)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'India — all {len(stations):,} railway stations')
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
plt.tight_layout()
plt.savefig('images/all_stations.png', dpi=150)
plt.show()

**Write:** Where are stations dense? Where are they sparse? Name one region that seems to have very few stations and suggest why.

*Your observation:*

### The connections between stations

We processed all 417,080 rows and found every consecutive stop pair. For each pair we counted how many different trains share that pair.

The result is saved as `track_edges_enriched.csv`. Every row means:
> *These two stations are directly connected by track, and this many trains run between them.*

In [ ]:
# Columns: station codes and names, state, number of trains, distance in km
track = pd.read_csv(DATA + 'track_edges_enriched.csv')

print(f'Total track connections: {len(track):,}')
print()
track.head(8)

### The trains

In [ ]:
trains = pd.read_csv(DATA + 'train_stats.csv')

print(f'Total trains: {len(trains):,}')
print()
print('What each column means:')
print('  n_stops           — how many stations this train halts at')
print('  total_distance_km — total route length (straight-line km between stops)')
print('  avg_km_per_stop   — average km between each halt')
print('  n_skip_segments   — how many times the train jumps over a station without stopping')
print()
trains.head(8)

---
## Part 2 — Express trains vs slow trains

`avg_km_per_stop` tells you how express a train is.

- A slow local train stopping every 5 km → `avg_km_per_stop = 5`  
- A Rajdhani stopping every 200 km → `avg_km_per_stop = 200`

**Before you compute:** Do you think trains that travel longer distances skip more stations? Does a Mumbai–Delhi train stop less often than a Delhi–Agra train?

*Your prediction (write before running the next cell):*

In [ ]:
print('Average km between consecutive halts — all trains:')
print(trains['avg_km_per_stop'].describe().round(1))
print()
print(f"Trains stopping every 10 km or less: {(trains['avg_km_per_stop'] <= 10).sum():,}")
print(f"Trains stopping every 30 km or more: {(trains['avg_km_per_stop'] >= 30).sum():,}")
print(f"Trains with ANY station skipped:     {(trains['n_skip_segments'] > 0).sum():,}")

In [ ]:
# Does a longer journey mean more skipping?
# Split trains into distance buckets and compare

short  = trains[trains['total_distance_km'] <  500]
medium = trains[(trains['total_distance_km'] >= 500) & (trains['total_distance_km'] < 1500)]
long   = trains[trains['total_distance_km'] >= 1500]

print(f"Short trains  (< 500 km):   {len(short):,} trains,  median {short['avg_km_per_stop'].median():.1f} km/stop")
print(f"Medium trains (500–1500 km): {len(medium):,} trains,  median {medium['avg_km_per_stop'].median():.1f} km/stop")
print(f"Long trains   (> 1500 km):  {len(long):,} trains,  median {long['avg_km_per_stop'].median():.1f} km/stop")
print()
print('Notice: the median is almost the same across all three groups.')
print('In India, even very long trains stop nearly as often as short ones.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(trains['total_distance_km'], trains['avg_km_per_stop'],
           s=3, alpha=0.2, color='steelblue')

# Highlight the genuine express outliers
express = trains[(trains['total_distance_km'] > 800) & (trains['avg_km_per_stop'] > 50)]
ax.scatter(express['total_distance_km'], express['avg_km_per_stop'],
           s=30, color='tomato', zorder=5, label=f'Genuine express ({len(express)} trains)')

ax.set_xlabel('Total route distance (km)')
ax.set_ylabel('Average km between consecutive halts')
ax.set_title('Most Indian trains cluster below 15 km/stop regardless of distance')
ax.legend()
plt.tight_layout()
plt.savefig('images/express_scatter.png', dpi=150)
plt.show()

**Write:** Were you right? What does it mean that even a 2000 km train stops every 7–8 km on average? What would that feel like as a passenger? What does it tell you about who Indian Railways serves?

*Your answer:*

---
## Part 3 — How do we build the physical track from timetables?

We connected every pair of consecutive stops. That gives us a lot of connections — but some are fake.

Look at the Howrah Rajdhani schedule again:

> New Delhi → **Kanpur** → Prayagraj → Mughal Sarai → Patna → Howrah

When we connect consecutive stops, we get the edge **New Delhi → Kanpur** (440 km).
That is a real track.

But another express train might go directly:
> New Delhi → Patna (skipping Kanpur and Prayagraj)

Now we have the edge **New Delhi → Patna** (1,000 km). That looks like a direct track — but it is not. The train simply did not stop in between.

**These fake long connections are called skip edges.** We need to remove them.

### How to spot a skip edge — the geometry trick

If station **B** lies geographically between stations **A** and **C**, then the connection A→C is almost certainly an express jump, not a real track.

The test is simple:

> If **distance(A to B)** + **distance(B to C)** ≈ **distance(A to C)**
> then B is "on the way" — the train passed through B without stopping.

Let us see this with a small example.

In [ ]:
# A tiny example: 5 connections on the Delhi–Patna corridor
toy = pd.DataFrame({
    'station_a':   ['New Delhi', 'New Delhi', 'Kanpur',   'Prayagraj', 'Mughal Sarai'],
    'station_b':   ['Kanpur',   'Patna',     'Prayagraj', 'Mughal Sarai', 'Patna'],
    'distance_km': [440,         1000,        190,         120,           250],
    'num_trains':  [80,          15,          90,          85,            100],
})

print('All consecutive stop pairs on this corridor:')
print(toy.to_string(index=False))
print()
print('New Delhi → Patna is 1000 km.')
print('But: Delhi→Kanpur (440) + Kanpur→...→Patna sums to 1000 km too.')
print('So Kanpur lies on the way. New Delhi → Patna is a SKIP EDGE — remove it.')

In [ ]:
# We ran this test on all 8,675 connections in the full dataset.
# Only 63 turned out to be skip edges (0.7%).
# The result — the physical track network — is already saved for you.

print('FULL DATASET RESULT:')
print(f'  Total consecutive stop pairs: 8,675')
print(f'  Skip edges found and removed: 63  (0.7%)')
print(f'  Physical track connections:   8,612')
print()
print('The track data you loaded earlier IS the physical network.')
print(f'It has {len(track):,} connections.')
print()
print('Surprising result: 99.3% of consecutive stop pairs are already real track.')
print('This connects directly to what you found in Part 2 — almost no train skips stations.')

**Write:** Why do you think so few skip edges exist? What does it tell you about how Indian Railways was designed — who is it built to serve?

*Your answer:*

---
## Part 4 — Five questions about Indian Railways

We have the full physical track network. Let us use it to answer five real questions.

### Question 1 — Which train makes the biggest single jump between consecutive stops?

Even after removing skip edges, some physical track stretches are very long — places where there genuinely is no station in between.

In [ ]:
# Sort by distance to find the longest physical track connections
longest = track.sort_values('distance_km', ascending=False)

print('10 longest physical track connections (no station in between):')
print()
print(f'{"From":<28} {"To":<28} {"Distance":>10} {"Trains":>8}')
print('-' * 76)
for _, row in longest.head(10).iterrows():
    print(f"{str(row['name_a']):<28} {str(row['name_b']):<28} {row['distance_km']:>8.0f} km {int(row['num_trains']):>7}")

**Write:** Look these places up on a map. What geographic features (sea, desert, mountains, border) explain why there is no station between them?

*Your answer:*

### Question 2 — How many stations are dead ends?

A dead end station has only one track connection. If that connection breaks — flood, landslide, broken bridge — the station has no way in or out.

We count connections per station by counting how many times each station appears in the track data.

In [ ]:
# Count how many track connections each station has
from_a = track.groupby('station_a').size()
from_b = track.groupby('station_b').size()
connections = from_a.add(from_b, fill_value=0).astype(int)

dead_ends    = connections[connections == 1]
pass_through = connections[connections == 2]
junctions    = connections[connections >= 3]

print(f'Dead end stations     (1 connection):   {len(dead_ends):,}')
print(f'Pass-through stations (2 connections):  {len(pass_through):,}')
print(f'Junction stations     (3+ connections): {len(junctions):,}')
print()
print(f'Fraction that are simple pass-throughs: {len(pass_through)/len(connections)*100:.0f}%')

In [ ]:
# Map the dead ends
coords = stations.set_index('code')[['lat','lon']]

dead_codes = dead_ends.index.tolist()
dead_locs  = coords.loc[coords.index.isin(dead_codes)]
all_locs   = coords

fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter(all_locs['lon'],  all_locs['lat'],  s=0.5, color='lightgrey', alpha=0.5, label='All stations')
ax.scatter(dead_locs['lon'], dead_locs['lat'], s=3,   color='tomato',    alpha=0.6, label=f'Dead ends ({len(dead_codes):,})')
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
ax.set_title('Dead end stations — one track connection only')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig('images/dead_ends.png', dpi=150)
plt.show()

**Write:** Where are dead end stations concentrated? What does this suggest about the strategy for expanding railways into remote or difficult terrain?

*Your answer:*

### Question 3 — How long is India's entire railway network?

In [ ]:
total_km = track['distance_km'].sum()

print(f'Total track length (straight-line): {total_km:,.0f} km')
print()
print('For comparison:')
print(f'  Circumference of Earth:  40,075 km')
print(f'  Our network is:          {total_km/40075*100:.0f}% of Earth circumference')
print()
print('Official Indian Railways figure: ~67,000 route km')
print(f'Our number ({total_km:,.0f} km) is lower because:')
print('  1. We measure straight-line distance between stations, not actual track curves')
print('  2. Some small branch lines may be missing from the schedule data')

### Question 4 — How far apart are stations from each other?

In [ ]:
avg_gap    = track['distance_km'].mean()
median_gap = track['distance_km'].median()

print(f'Average gap between neighboring stations: {avg_gap:.1f} km')
print(f'Median gap:                               {median_gap:.1f} km')
print()

# Which states have the densest and thinnest coverage?
by_state = track.groupby('state_a')['distance_km'].median()
by_state = by_state[track.groupby('state_a').size() >= 20]  # only states with enough data
by_state = by_state.sort_values()

print('States with densest rail coverage (smallest gap between stations):')
print(by_state.head(6).round(1).to_string())
print()
print('States with thinnest rail coverage (largest gap):')
print(by_state.tail(6).round(1).to_string())

### Question 5 — Which stretch of track carries the most trains?

In [ ]:
busiest = track.sort_values('num_trains', ascending=False)

print('15 busiest track connections (most trains per day):')
print()
print(f'{"From":<26} {"To":<26} {"Trains":>8} {"Distance":>10}')
print('-' * 72)
for _, row in busiest.head(15).iterrows():
    print(f"{str(row['name_a']):<26} {str(row['name_b']):<26} {int(row['num_trains']):>8} {row['distance_km']:>8.1f} km")

print()
print(f"Average distance for the 15 busiest connections: {busiest.head(15)['distance_km'].mean():.1f} km")
print('Observation: the busiest corridors are all very short — suburban commuter lines.')

**Write:** The busiest stretches of track are all very short — under 10 km. Why? What city are most of them near? What does this tell you about how most people actually use Indian Railways day to day?

*Your answer:*

---
## What we built today

From train timetables we:
- Understood what the raw schedule data contains
- Discovered that almost every Indian train stops at every station (genuine express trains are rare)
- Built the physical track network by removing express jumps using geometry
- Answered five real questions about India's railway system

**One question to think about before next session:**

If you had to shut down one station to cause the maximum disruption across the whole country — not the busiest station, not the most famous — which would you choose? And why might the answer be a station most people have never heard of?

**Next session:** Notebook 2 — Who Matters?